# 02. Operation Scenario and Vacuum Fields

A tokamak discharge does not simply happen. Coils are charged, gas is admitted, a loop
voltage is induced, and if the field at the moment of breakdown has the right shape, a
plasma forms and carries current. This session reads one VEST discharge backwards: from the
signals that show what the plasma did, to the fields that let it start at all.


## Session Overview

By the end of this session you will be able to:

- read the plasma-response signals of a discharge and say when the plasma formed;
- separate the current in the machine into what the coils drove, what the vessel carried,
  and what the plasma did;
- compute the vacuum quantities that decide whether a startup can work at all — the loop
  voltage, the vertical field, and the field decay index;
- check that the vacuum-field model reproduces the magnetics actually measured, before
  trusting anything derived from it;
- find the breakdown time from the data rather than by eye.

Everything runs offline on the packaged VEST discharge 39915.


## Physical Context

### Getting a plasma started

Startup has three phases, and each leaves a different fingerprint in the diagnostics.

**Breakdown.** The central solenoid swings, inducing a toroidal electric field. Free
electrons accelerate, ionise the fill gas, and an avalanche runs away. This needs the
electric field to be strong enough *and* the magnetic field lines to be long enough that an
electron gains energy before it reaches a wall — which is why the field null matters.

**Burn-through.** The young plasma is cold and full of neutrals and impurities, which radiate
away much of the ohmic input. It either heats through this barrier or it dies. Line
radiation, which the filterscope sees, is the direct evidence.

**Current ramp-up.** Past burn-through the plasma is conducting well, and the transformer
drives its current up.

### The vacuum field, and why it is a *precondition*

Before any plasma exists, the coils and the vessel already make a field. Three properties of
it decide whether startup is possible:

- **the loop voltage** $V_{\text{loop}} = -\,\mathrm{d}\psi/\mathrm{d}t$, the drive;
- **the vertical field** $B_z$, which must balance the plasma's outward hoop force once
  current flows;
- **the decay index** $n = -\dfrac{R}{B_z}\dfrac{\partial B_z}{\partial R}$, which decides
  whether that balance is *stable*. For a plasma to be passively stable against vertical
  displacement, $0 < n < 1.5$.

### The vessel is part of the circuit

VEST's vacuum vessel is a conductor. A changing coil current induces eddy currents in it,
and those currents make their own field, delayed and smeared relative to the coils. Any
honest vacuum-field calculation has to include them, which is why this session solves for
them before computing anything else.


## Load / Prepare Data

### The discharge


In [ ]:
import numpy as np
import vaft
import matplotlib.pyplot as plt


In [ ]:
ods = vaft.omas.sample_ods()
sorted(ods.keys())


### Solving the vessel currents

The packaged shot has 950 passive loops describing the vessel, but no currents in them —
solving that circuit is a modelling step, not a measurement, so it is not stored.

Passing empty plasma filaments computes the **vacuum** case: the vessel's response to the
coils alone, with no plasma. That is exactly the field a startup has to work with.


In [ ]:
names = {row["name"] for row in vaft.omas.available_plots(ods)}
print("passive current available?", "passive_structure_time_current" in names)
print("pf_passive loops        :", len(ods["pf_passive.loop"]))
print("pf_passive time         :", "time" in ods["pf_passive"])


In [ ]:
vaft.omas.compute_eddy_currents(ods, [], [])

unlocked = {row["name"] for row in vaft.omas.available_plots(ods)} - names
print("time points solved:", len(ods["pf_passive.time"]))
print("plots unlocked    :", sorted(unlocked))


Three views appeared, and the reason is worth noticing: none of them is about the vessel
itself. Two are magnetics comparisons and one is the vessel current. They became available
because the vacuum-field model they rest on now exists.


## Guided Analysis

### What the plasma did

Start with the response signals. The plasma current says whether there was a plasma; the
line emission says what state it was in.


In [ ]:
vaft.omas.plot_plasma_current_time(ods)
plt.show()


In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha")
plt.show()


H-alpha is neutral hydrogen at the edge — recycling and fuelling. Impurity lines tell the
burn-through story instead: carbon and oxygen come off the wall, radiate, and have to be
overcome.


In [ ]:
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="CIII")
plt.show()


### What drove it

Two coil systems. The toroidal field coil sets the field the plasma sits in; the poloidal
field coils drive and shape it.


In [ ]:
vaft.omas.plot_tf_coil_time_current(ods)
plt.show()


In [ ]:
vaft.omas.plot_pf_coil_time_current(ods)
plt.show()


Two other actuators belong in this picture and are not here: the **gas valve command** and
the **EC pre-ionisation trigger**. Both are recorded on VEST but neither is mapped into IMAS
yet, so there is nothing to plot. Their absence is a mapping gap, not a quiet discharge.

### Where the current actually is

At any moment the machine carries current in three places, and only one of them is the
plasma. Separating them is the first real analysis step of the session.


In [ ]:
vaft.omas.plot_current_overview(ods)
plt.show()


The vessel current is large — comparable to the plasma current — and it is *induced*, so it
opposes whatever the coils just did. A magnetic measurement made outside the vessel sees the
sum of all three. That is the whole difficulty of magnetic reconstruction in one figure.

### Does the vacuum model match the machine?

Everything below this point is derived from the field model. Before trusting it, check it
against the magnetics that were actually measured.


In [ ]:
vaft.omas.plot_magnetics_overview_vacuum(ods)
plt.show()


### The drive

The loop voltage is the time derivative of poloidal flux at a point — literally what the
transformer is doing to the plasma's future location.


In [ ]:
time, v_loop = vaft.omas.compute_startup_loop_voltage_ods(ods, rz=(0.4, 0.0))

figure, axes = plt.subplots()
axes.plot(time, v_loop)
axes.set_xlabel("Time [s]")
axes.set_ylabel(r"$V_{\mathrm{loop}}$ [V]")
axes.grid(alpha=0.3)
plt.show()

print(f"peak |V_loop| = {np.nanmax(np.abs(v_loop)):.2f} V "
      f"at t = {time[np.nanargmax(np.abs(v_loop))]:.4f} s")


### The vertical field and its decay index

A plasma ring wants to expand. The vertical field pushes back, and whether that restoring
force is *stable* depends on how fast the field falls off with radius.


In [ ]:
radius, decay_index = vaft.omas.compute_decay_index_ods(ods, time=0.3307)

figure, axes = plt.subplots()
axes.plot(radius, decay_index)
axes.axhspan(0.0, 1.5, alpha=0.15, label="passively stable")
axes.set_xlabel("R [m]")
axes.set_ylabel("decay index $n$")
axes.legend()
axes.grid(alpha=0.3)
plt.show()

finite = np.isfinite(decay_index)
print(f"n spans {decay_index[finite].min():.2f} to {decay_index[finite].max():.2f}")
print("entirely inside 0 < n < 1.5:",
      bool(np.all((decay_index[finite] > 0.0) & (decay_index[finite] < 1.5))))


The index is inside the stable band across the whole radial range, so a plasma formed
anywhere here would be held rather than flung out. Note that $n$ is undefined where $B_z$
crosses zero — the calculation returns `nan` there rather than a large meaningless number.

### When did the plasma form?

Reading a breakdown time off an H-alpha trace by eye is a habit worth breaking. The onset can
be found from the data.


In [ ]:
t_breakdown = vaft.omas.find_breakdown_onset(ods)
print(f"breakdown onset: {t_breakdown * 1e3:.2f} ms")


In [ ]:
vaft.omas.plot_magnetics_overview_plasma_residual(ods)
plt.show()


### The field null

Breakdown needs somewhere for electrons to accelerate without promptly hitting a wall — a
region of weak poloidal field with long connection lengths. In the vacuum flux map it shows
up as the null.


In [ ]:
vaft.omas.plot_equilibrium_field_psi_vacuum(ods, time=0.3307)
plt.show()


## Interpretation Checkpoints

Each of these has a definite answer in what you have already plotted.

1. **Compare the breakdown time with the H-alpha rise.** Do they agree? Which would you
   trust to define "the plasma started", and why?
2. **The vessel current opposes the coil current.** Look at the three-way decomposition:
   at what point in the discharge is the vessel contribution largest, and what is the
   transformer doing then?
3. **The loop voltage peaks before the plasma current does.** Should it? What does the delay
   between them tell you about the plasma's inductance?
4. **The decay index sits inside the stable band everywhere.** What would the figure look
   like for a machine where it did not, and what would happen to the plasma?
5. **`compute_eddy_currents` unlocked the magnetics comparison plots.** Why should a
   *modelling* step change what you are allowed to plot?


## Integrated Analysis

### Was this a good startup?

You now have every piece needed to answer that as a physicist rather than by impression.
Assemble them on one time axis and read the sequence.


In [ ]:
figure, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 8))

vaft.omas.plot_plasma_current_time(ods, ax=axes[0])
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha", ax=axes[1])
axes[2].plot(time, v_loop)
axes[2].set_ylabel(r"$V_{\mathrm{loop}}$ [V]")
axes[2].grid(alpha=0.3)

for axis in axes:
    axis.axvline(t_breakdown, color="k", ls="--", lw=0.8)
axes[2].set_xlabel("Time [s]")
plt.show()


The dashed line is the breakdown time found from the data. Read left to right: the loop
voltage is applied first, the plasma forms, H-alpha rises as the young plasma recycles
against the wall, and the current ramps.

The story this discharge tells is a startup that worked, in a vacuum field that was stable
everywhere it could have formed.


## Independent Exercise

### Reconstruct another shot's operating conditions

Take a different VEST discharge and repeat this workflow: solve the vessel currents, check
the vacuum model against the magnetics, find the breakdown time, and decide from the decay
index whether the field it formed in was stable. Then say, in a sentence, what kind of
startup it was.

Loading another shot needs database access, so the cells below are left for you to run in
lab mode.

### Two things this session cannot yet compute

Issue [#230](https://github.com/VEST-Tokamak/vaft/issues/230) asks for two more quantities.
Neither exists in VAFT today, and it is better to say so than to hand you an approximation
dressed as a result.

**Connection length.** How far a field line travels before striking a surface, which is what
actually decides whether an electron avalanches during breakdown. VAFT can trace field lines
(`vaft.process.equilibrium.trace_field_line`) but has no connection-length calculation on top
of that, and no wall-intersection test to terminate one.

**The 2.45 GHz EC resonance layer.** Where the electron cyclotron frequency matches the
pre-ionisation source, given the toroidal field. This is a one-line calculation from
$B_t(R)$ — but VEST's diagnostic registry describes no 2.45 GHz system at all, so there is
nothing to place the layer *for*. That looks like a mapping gap rather than a physics one,
and is worth confirming before anyone implements it.


In [ ]:
# 1. Load another discharge. Needs HSDS access, so this is lab mode.
# other = vaft.database.load(41672)

# 2. Solve its vessel currents in the vacuum case.
# vaft.omas.compute_eddy_currents(other, [], [])

# 3. Check the model against the measured magnetics before trusting it.
# vaft.omas.plot_magnetics_overview_vacuum(other)
# plt.show()

# 4. Find the breakdown time and the decay index at that moment.
# t_bd = vaft.omas.find_breakdown_onset(other)
# radius, n_index = vaft.omas.compute_decay_index_ods(other, time=t_bd)

# 5. Was the field it formed in stable? Compare with 39915.


## Takeaways and Next Steps

- A discharge is a sequence — breakdown, burn-through, ramp-up — and each phase shows up in
  a different diagnostic. Reading them together is what makes a scenario legible.
- The vessel is part of the circuit. Vessel currents are comparable to the plasma current and
  oppose the coils, and every magnetic measurement sees their sum.
- A vacuum field is a *precondition*, not an afterthought: the loop voltage supplies the
  drive, $B_z$ supplies the balance, and the decay index decides whether that balance holds.
- Check a model against what was measured before deriving anything from it. That is what
  `magnetics_overview_vacuum` is for, and it comes before the physics, not after.
- When a quantity cannot be computed, say so. Two of this session's intended exercises are
  named above rather than approximated.

**Next**: Session 03 takes the discharge past startup and reconstructs the equilibrium it
settled into.
